# 14 Registro no rigido por mascaras - todos los sujetos

Este notebook prueba el primer registro deformable H&E -> HSI.

Como `SimpleITK` no esta instalado en este entorno, usamos `scikit-image` con **flujo optico denso TV-L1** sobre mapas de distancia firmados. En la practica, para este caso cumple un papel parecido a Demons: estima un campo de desplazamiento suave para deformar localmente la H&E prealineada hacia la HSI.

Flujo general:

1. Cargar HSI fija desde `Imagenes_a_escala`.
2. Cargar la H&E ya prealineada desde `00_baseline_clasico/outputs/outputs_registro_afin_mascaras`.
3. Convertir ambas mascaras a mapas de distancia firmados.
4. Estimar un campo denso no rigido.
5. Aplicar el campo a la H&E RGB y a su mascara.
6. Comparar contra el mejor pre-registro rigido/afin.

El objetivo aqui no es solo subir Dice: tambien miramos la magnitud del campo de deformacion para detectar resultados poco realistas.


In [ ]:
from pathlib import Path
import csv
import json
import math

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from scipy.ndimage import gaussian_filter
from skimage.registration import optical_flow_tvl1


BASE_DIR = Path(r'Registration\DeepLearning')
INPUT_DIR = BASE_DIR / 'Imagenes_a_escala'
PREALIGN_ROOT = BASE_DIR / 'Tecnicas_registration' / '00_baseline_clasico' / 'outputs' / 'outputs_registro_afin_mascaras'
OUTPUT_ROOT = BASE_DIR / 'Tecnicas_registration' / '00_baseline_clasico' / 'outputs' / 'outputs_registro_no_rigido_mascaras'
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

SUBJECTS = ['SB012', 'SB013', 'SB017', 'SB018', 'SB019', 'SB020']

print('Entrada base:', INPUT_DIR)
print('Pre-registro:', PREALIGN_ROOT)
print('Salida:', OUTPUT_ROOT)


## Carga de datos

La HSI es la imagen fija. La H&E viene ya transformada por el mejor paso anterior:

- si el afín fue aceptado, usamos afín;
- si no, ese archivo ya contiene el rígido final.


In [ ]:
def load_rgb(path):
    return np.asarray(Image.open(path).convert('RGB'), dtype=np.uint8)


def load_mask(path):
    return np.asarray(Image.open(path).convert('L'), dtype=np.uint8) > 0


def subject_paths(subject_id):
    subject_dir = INPUT_DIR / subject_id
    prealign_dir = PREALIGN_ROOT / subject_id
    return {
        'hsi_rgb': subject_dir / f'{subject_id}_hsi.png',
        'hsi_mask': subject_dir / f'{subject_id}_hsi_mask.png',
        'he_prealigned_rgb': prealign_dir / f'{subject_id}_he_affine_to_hsi.png',
        'he_prealigned_mask': prealign_dir / f'{subject_id}_he_mask_affine_to_hsi.png',
        'affine_metrics': prealign_dir / f'{subject_id}_affine_metrics.json',
    }


## Metricas y visualizacion

In [ ]:
def dice_score(mask_a, mask_b):
    a = np.asarray(mask_a) > 0
    b = np.asarray(mask_b) > 0
    denom = a.sum() + b.sum()
    if denom == 0:
        return 1.0
    return float(2.0 * np.logical_and(a, b).sum() / denom)


def iou_score(mask_a, mask_b):
    a = np.asarray(mask_a) > 0
    b = np.asarray(mask_b) > 0
    union = np.logical_or(a, b).sum()
    if union == 0:
        return 1.0
    return float(np.logical_and(a, b).sum() / union)


def contour_mask(mask, thickness=1):
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8) * 255
    contours, _ = cv2.findContours(mask_u8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_NONE)
    out = np.zeros_like(mask_u8)
    cv2.drawContours(out, contours, -1, 255, thickness=int(thickness))
    return out > 0


def symmetric_contour_distance(mask_a, mask_b):
    contour_a = contour_mask(mask_a)
    contour_b = contour_mask(mask_b)
    if not contour_a.any() or not contour_b.any():
        return {'mean': float('nan'), 'p95': float('nan')}

    dist_to_b = cv2.distanceTransform((~contour_b).astype(np.uint8), cv2.DIST_L2, 5)
    dist_to_a = cv2.distanceTransform((~contour_a).astype(np.uint8), cv2.DIST_L2, 5)
    distances = np.concatenate([dist_to_b[contour_a], dist_to_a[contour_b]])
    return {
        'mean': float(np.mean(distances)),
        'p95': float(np.percentile(distances, 95)),
    }


def normalize_u8(gray):
    arr = np.asarray(gray, dtype=np.float32)
    lo, hi = np.percentile(arr, [1, 99])
    if hi <= lo:
        return np.zeros(arr.shape, dtype=np.uint8)
    return (np.clip((arr - lo) / (hi - lo), 0, 1) * 255).astype(np.uint8)


def overlay_rgb_images(fixed_rgb, moving_rgb, fixed_mask=None, moving_mask=None):
    fixed_gray = cv2.cvtColor(np.asarray(fixed_rgb, dtype=np.uint8), cv2.COLOR_RGB2GRAY)
    moving_gray = cv2.cvtColor(np.asarray(moving_rgb, dtype=np.uint8), cv2.COLOR_RGB2GRAY)
    fixed_norm = normalize_u8(fixed_gray)
    moving_norm = normalize_u8(moving_gray)

    overlay = np.zeros((*fixed_gray.shape, 3), dtype=np.uint8)
    overlay[..., 0] = fixed_norm
    overlay[..., 1] = moving_norm
    overlay[..., 2] = fixed_norm // 3

    if fixed_mask is not None:
        overlay[contour_mask(fixed_mask, thickness=2)] = np.array([255, 0, 0], dtype=np.uint8)
    if moving_mask is not None:
        overlay[contour_mask(moving_mask, thickness=2)] = np.array([0, 255, 255], dtype=np.uint8)
    return overlay


def contours_overlay_image(fixed_mask, moving_mask):
    out = np.zeros((*fixed_mask.shape, 3), dtype=np.uint8)
    out[contour_mask(fixed_mask, thickness=2)] = np.array([255, 0, 0], dtype=np.uint8)
    out[contour_mask(moving_mask, thickness=2)] = np.array([0, 255, 255], dtype=np.uint8)
    return out


def save_mask(path, mask):
    Image.fromarray((np.asarray(mask) > 0).astype(np.uint8) * 255).save(path)


def write_csv(path, rows, fieldnames):
    with path.open('w', newline='', encoding='utf-8') as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in rows:
            writer.writerow({key: row.get(key, '') for key in fieldnames})


## Mapas de distancia y campo no rigido

No registramos colores. Registramos forma.

El mapa de distancia firmado da una imagen continua:

- valores altos dentro del tejido,
- valores bajos fuera,
- transicion suave alrededor del contorno.

Esto suele ser mas estable que optimizar directamente una mascara binaria.


In [ ]:
def signed_distance_field(mask, clip_percentile=98):
    mask_u8 = (np.asarray(mask) > 0).astype(np.uint8)
    inside = cv2.distanceTransform(mask_u8, cv2.DIST_L2, 5)
    outside = cv2.distanceTransform(1 - mask_u8, cv2.DIST_L2, 5)
    signed = inside - outside
    clip = float(np.percentile(np.abs(signed), clip_percentile))
    if not np.isfinite(clip) or clip <= 0:
        clip = 1.0
    signed = np.clip(signed, -clip, clip)
    signed = (signed + clip) / (2.0 * clip)
    return signed.astype(np.float32)


def estimate_dense_flow(fixed_mask, moving_mask):
    fixed_field = signed_distance_field(fixed_mask)
    moving_field = signed_distance_field(moving_mask)

    flow = optical_flow_tvl1(
        fixed_field,
        moving_field,
        attachment=10,
        tightness=0.3,
        num_warp=8,
        num_iter=80,
        tol=1e-4,
        prefilter=True,
        dtype=np.float32,
    )

    # Suavizado ligero extra para evitar campos con pequenos dientes en el contorno.
    flow_smooth = np.empty_like(flow, dtype=np.float32)
    flow_smooth[0] = gaussian_filter(flow[0], sigma=1.2)
    flow_smooth[1] = gaussian_filter(flow[1], sigma=1.2)
    return flow_smooth, fixed_field, moving_field


def warp_dense(image, flow, sign=1.0, is_mask=False):
    h, w = image.shape[:2]
    grid_y, grid_x = np.mgrid[0:h, 0:w].astype(np.float32)
    map_y = grid_y + sign * flow[0].astype(np.float32)
    map_x = grid_x + sign * flow[1].astype(np.float32)

    if is_mask:
        src = (np.asarray(image) > 0).astype(np.uint8) * 255
        warped = cv2.remap(
            src,
            map_x,
            map_y,
            interpolation=cv2.INTER_NEAREST,
            borderMode=cv2.BORDER_CONSTANT,
            borderValue=0,
        )
        return warped > 0

    return cv2.remap(
        np.asarray(image, dtype=np.uint8),
        map_x,
        map_y,
        interpolation=cv2.INTER_LINEAR,
        borderMode=cv2.BORDER_CONSTANT,
        borderValue=0,
    )


def flow_diagnostics(flow):
    mag = np.sqrt(flow[0] ** 2 + flow[1] ** 2)
    return {
        'mean': float(np.mean(mag)),
        'p50': float(np.percentile(mag, 50)),
        'p95': float(np.percentile(mag, 95)),
        'max': float(np.max(mag)),
    }


def flow_magnitude_image(flow):
    mag = np.sqrt(flow[0] ** 2 + flow[1] ** 2)
    mag_u8 = normalize_u8(mag)
    return cv2.applyColorMap(mag_u8, cv2.COLORMAP_TURBO)[:, :, ::-1]


## Criterio de aceptacion

Este paso es delicado: un campo deformable puede mejorar Dice demasiado facil si aplasta una mascara. Por eso el resultado final se acepta solo si:

- mejora Dice respecto al pre-registro,
- mejora la distancia media entre contornos,
- el desplazamiento p95 y maximo no son exagerados.

Aun asi, siempre guardamos tambien el candidato para revisarlo visualmente.


In [ ]:
def choose_best_flow_sign(flow, moving_mask, fixed_mask):
    candidates = []
    for sign in [1.0, -1.0]:
        warped_mask = warp_dense(moving_mask, flow, sign=sign, is_mask=True)
        contour_distance = symmetric_contour_distance(warped_mask, fixed_mask)
        candidates.append({
            'sign': sign,
            'mask': warped_mask,
            'dice': dice_score(warped_mask, fixed_mask),
            'iou': iou_score(warped_mask, fixed_mask),
            'contour_mean': contour_distance['mean'],
            'contour_p95': contour_distance['p95'],
        })
    return max(candidates, key=lambda row: (row['dice'], -row['contour_mean']))


def accept_nonrigid(pre_dice, pre_contour_mean, candidate_metrics, flow_diag):
    dice_gain = candidate_metrics['dice'] - pre_dice
    contour_gain = pre_contour_mean - candidate_metrics['contour_mean']

    if dice_gain < 0.003:
        return False, 'rejected_low_dice_gain'
    if contour_gain < 1.0:
        return False, 'rejected_low_contour_gain'
    if flow_diag['p95'] > 90:
        return False, 'rejected_p95_displacement_too_large'
    if flow_diag['max'] > 180:
        return False, 'rejected_max_displacement_too_large'
    return True, 'nonrigid_accepted'


## Procesamiento por sujeto

In [ ]:
def process_subject(subject_id):
    paths = subject_paths(subject_id)
    fixed_rgb = load_rgb(paths['hsi_rgb'])
    fixed_mask = load_mask(paths['hsi_mask'])
    moving_rgb_pre = load_rgb(paths['he_prealigned_rgb'])
    moving_mask_pre = load_mask(paths['he_prealigned_mask'])
    affine_metrics = json.loads(paths['affine_metrics'].read_text(encoding='utf-8'))

    out_dir = OUTPUT_ROOT / subject_id
    out_dir.mkdir(parents=True, exist_ok=True)

    pre_dice = dice_score(moving_mask_pre, fixed_mask)
    pre_iou = iou_score(moving_mask_pre, fixed_mask)
    pre_contour = symmetric_contour_distance(moving_mask_pre, fixed_mask)

    flow, fixed_field, moving_field = estimate_dense_flow(fixed_mask, moving_mask_pre)
    flow_diag = flow_diagnostics(flow)
    best_candidate = choose_best_flow_sign(flow, moving_mask_pre, fixed_mask)
    accepted, decision = accept_nonrigid(pre_dice, pre_contour['mean'], best_candidate, flow_diag)

    if accepted:
        final_mask = best_candidate['mask']
        final_rgb = warp_dense(moving_rgb_pre, flow, sign=best_candidate['sign'], is_mask=False)
    else:
        final_mask = moving_mask_pre
        final_rgb = moving_rgb_pre

    final_contour = symmetric_contour_distance(final_mask, fixed_mask)
    final_dice = dice_score(final_mask, fixed_mask)
    final_iou = iou_score(final_mask, fixed_mask)

    candidate_rgb = warp_dense(moving_rgb_pre, flow, sign=best_candidate['sign'], is_mask=False)
    overlay_final = overlay_rgb_images(fixed_rgb, final_rgb, fixed_mask, final_mask)
    contours_final = contours_overlay_image(fixed_mask, final_mask)
    overlay_candidate = overlay_rgb_images(fixed_rgb, candidate_rgb, fixed_mask, best_candidate['mask'])
    contours_candidate = contours_overlay_image(fixed_mask, best_candidate['mask'])
    flow_mag_rgb = flow_magnitude_image(flow)

    files = {
        'he_rgb_nonrigid': out_dir / f'{subject_id}_he_nonrigid_to_hsi.png',
        'he_mask_nonrigid': out_dir / f'{subject_id}_he_mask_nonrigid_to_hsi.png',
        'he_rgb_candidate': out_dir / f'{subject_id}_he_nonrigid_candidate_to_hsi.png',
        'he_mask_candidate': out_dir / f'{subject_id}_he_mask_nonrigid_candidate_to_hsi.png',
        'overlay_rgb': out_dir / f'{subject_id}_overlay_rgb_nonrigid.png',
        'overlay_contours': out_dir / f'{subject_id}_overlay_contours_nonrigid.png',
        'overlay_rgb_candidate': out_dir / f'{subject_id}_overlay_rgb_nonrigid_candidate.png',
        'overlay_contours_candidate': out_dir / f'{subject_id}_overlay_contours_nonrigid_candidate.png',
        'flow_magnitude': out_dir / f'{subject_id}_flow_magnitude.png',
        'flow_npz': out_dir / f'{subject_id}_dense_flow.npz',
        'metrics': out_dir / f'{subject_id}_nonrigid_metrics.json',
    }

    Image.fromarray(final_rgb).save(files['he_rgb_nonrigid'])
    save_mask(files['he_mask_nonrigid'], final_mask)
    Image.fromarray(candidate_rgb).save(files['he_rgb_candidate'])
    save_mask(files['he_mask_candidate'], best_candidate['mask'])
    Image.fromarray(overlay_final).save(files['overlay_rgb'])
    Image.fromarray(contours_final).save(files['overlay_contours'])
    Image.fromarray(overlay_candidate).save(files['overlay_rgb_candidate'])
    Image.fromarray(contours_candidate).save(files['overlay_contours_candidate'])
    Image.fromarray(flow_mag_rgb).save(files['flow_magnitude'])
    np.savez_compressed(
        files['flow_npz'],
        flow_y=flow[0].astype(np.float32),
        flow_x=flow[1].astype(np.float32),
        selected_sign=np.array(best_candidate['sign'], dtype=np.float32),
        fixed_field=fixed_field.astype(np.float32),
        moving_field=moving_field.astype(np.float32),
    )

    metrics = {
        'subject_id': subject_id,
        'registration_direction': 'he_to_hsi',
        'fixed_image': 'hsi',
        'moving_image': 'he',
        'prealignment_source': 'outputs_registro_afin_mascaras',
        'prealignment_affine_accepted': affine_metrics.get('affine_accepted'),
        'prealignment_note': affine_metrics.get('affine_final_note'),
        'nonrigid_method': 'skimage_optical_flow_tvl1_on_signed_distance_maps',
        'nonrigid_accepted': accepted,
        'nonrigid_decision': decision,
        'selected_flow_sign': float(best_candidate['sign']),
        'dice_pre': pre_dice,
        'dice_nonrigid_candidate': best_candidate['dice'],
        'dice_nonrigid_final': final_dice,
        'iou_pre': pre_iou,
        'iou_nonrigid_candidate': best_candidate['iou'],
        'iou_nonrigid_final': final_iou,
        'contour_mean_distance_pre_px': pre_contour['mean'],
        'contour_mean_distance_candidate_px': best_candidate['contour_mean'],
        'contour_mean_distance_final_px': final_contour['mean'],
        'contour_p95_distance_pre_px': pre_contour['p95'],
        'contour_p95_distance_candidate_px': best_candidate['contour_p95'],
        'contour_p95_distance_final_px': final_contour['p95'],
        'flow_mean_displacement_px': flow_diag['mean'],
        'flow_p50_displacement_px': flow_diag['p50'],
        'flow_p95_displacement_px': flow_diag['p95'],
        'flow_max_displacement_px': flow_diag['max'],
        'overlay_rgb_path': str(files['overlay_rgb']),
        'overlay_contours_path': str(files['overlay_contours']),
        'overlay_rgb_candidate_path': str(files['overlay_rgb_candidate']),
        'overlay_contours_candidate_path': str(files['overlay_contours_candidate']),
        'flow_magnitude_path': str(files['flow_magnitude']),
        'flow_npz_path': str(files['flow_npz']),
    }
    files['metrics'].write_text(json.dumps(metrics, indent=2), encoding='utf-8')
    return metrics, overlay_final, contours_final, overlay_candidate, contours_candidate


## Ejecutar todos los sujetos

In [ ]:
all_metrics = []
overlay_images = {}
contour_images = {}
candidate_overlay_images = {}
candidate_contour_images = {}

for subject_id in SUBJECTS:
    print(f'=== {subject_id} ===')
    metrics, overlay, contours, overlay_candidate, contours_candidate = process_subject(subject_id)
    all_metrics.append(metrics)
    overlay_images[subject_id] = overlay
    contour_images[subject_id] = contours
    candidate_overlay_images[subject_id] = overlay_candidate
    candidate_contour_images[subject_id] = contours_candidate
    print(
        f"pre Dice={metrics['dice_pre']:.4f} -> "
        f"candidate={metrics['dice_nonrigid_candidate']:.4f} -> "
        f"final={metrics['dice_nonrigid_final']:.4f} | "
        f"accepted={metrics['nonrigid_accepted']} | "
        f"{metrics['nonrigid_decision']} | "
        f"flow p95={metrics['flow_p95_displacement_px']:.1f}px"
    )

summary_csv = OUTPUT_ROOT / 'nonrigid_mask_summary.csv'
summary_json = OUTPUT_ROOT / 'nonrigid_mask_summary.json'
summary_fields = [
    'subject_id',
    'prealignment_affine_accepted',
    'prealignment_note',
    'nonrigid_accepted',
    'nonrigid_decision',
    'dice_pre',
    'dice_nonrigid_candidate',
    'dice_nonrigid_final',
    'iou_pre',
    'iou_nonrigid_candidate',
    'iou_nonrigid_final',
    'contour_mean_distance_pre_px',
    'contour_mean_distance_candidate_px',
    'contour_mean_distance_final_px',
    'contour_p95_distance_pre_px',
    'contour_p95_distance_candidate_px',
    'contour_p95_distance_final_px',
    'flow_mean_displacement_px',
    'flow_p95_displacement_px',
    'flow_max_displacement_px',
    'selected_flow_sign',
    'overlay_rgb_path',
    'overlay_contours_path',
    'overlay_rgb_candidate_path',
    'overlay_contours_candidate_path',
    'flow_magnitude_path',
]
write_csv(summary_csv, all_metrics, summary_fields)
summary_json.write_text(json.dumps(all_metrics, indent=2), encoding='utf-8')

print('Resumen CSV:', summary_csv)
print('Resumen JSON:', summary_json)


## Resumen numerico

In [ ]:
for row in all_metrics:
    delta_final = row['dice_nonrigid_final'] - row['dice_pre']
    delta_candidate = row['dice_nonrigid_candidate'] - row['dice_pre']
    print(
        f"{row['subject_id']}: "
        f"Dice pre={row['dice_pre']:.4f}, "
        f"candidate={row['dice_nonrigid_candidate']:.4f} ({delta_candidate:+.4f}), "
        f"final={row['dice_nonrigid_final']:.4f} ({delta_final:+.4f}) | "
        f"contour pre={row['contour_mean_distance_pre_px']:.2f}px, "
        f"candidate={row['contour_mean_distance_candidate_px']:.2f}px, "
        f"final={row['contour_mean_distance_final_px']:.2f}px | "
        f"flow p95={row['flow_p95_displacement_px']:.1f}px | "
        f"{row['nonrigid_decision']}"
    )


## Montajes visuales

In [ ]:
def resize_for_montage(img, target_width=360):
    h, w = img.shape[:2]
    scale = target_width / max(w, 1)
    target_h = max(1, int(round(h * scale)))
    return cv2.resize(img, (target_width, target_h), interpolation=cv2.INTER_AREA)


def add_label(img, label):
    out = img.copy()
    cv2.rectangle(out, (0, 0), (out.shape[1], 30), (0, 0, 0), -1)
    cv2.putText(out, label, (8, 21), cv2.FONT_HERSHEY_SIMPLEX, 0.50, (255, 255, 255), 1, cv2.LINE_AA)
    return out


def make_grid(images, labels, cols=3, target_width=360):
    resized = [add_label(resize_for_montage(img, target_width), label) for img, label in zip(images, labels)]
    rows = int(math.ceil(len(resized) / cols))
    max_h = max(img.shape[0] for img in resized)
    cell_w = target_width
    canvas = np.zeros((rows * max_h, cols * cell_w, 3), dtype=np.uint8)
    for idx, img in enumerate(resized):
        r = idx // cols
        c = idx % cols
        y = r * max_h
        x = c * cell_w
        canvas[y:y + img.shape[0], x:x + img.shape[1]] = img
    return canvas


labels = [
    f"{row['subject_id']} | Dice {row['dice_nonrigid_final']:.3f} | {'nonrigid' if row['nonrigid_accepted'] else 'pre'}"
    for row in all_metrics
]
candidate_labels = [
    f"{row['subject_id']} cand | Dice {row['dice_nonrigid_candidate']:.3f} | p95 {row['flow_p95_displacement_px']:.0f}px"
    for row in all_metrics
]

overlay_grid = make_grid([overlay_images[sid] for sid in SUBJECTS], labels)
contour_grid = make_grid([contour_images[sid] for sid in SUBJECTS], labels)
candidate_overlay_grid = make_grid([candidate_overlay_images[sid] for sid in SUBJECTS], candidate_labels)
candidate_contour_grid = make_grid([candidate_contour_images[sid] for sid in SUBJECTS], candidate_labels)

overlay_grid_path = OUTPUT_ROOT / 'nonrigid_mask_overlay_grid.png'
contour_grid_path = OUTPUT_ROOT / 'nonrigid_mask_contour_grid.png'
candidate_overlay_grid_path = OUTPUT_ROOT / 'nonrigid_mask_candidate_overlay_grid.png'
candidate_contour_grid_path = OUTPUT_ROOT / 'nonrigid_mask_candidate_contour_grid.png'

Image.fromarray(overlay_grid).save(overlay_grid_path)
Image.fromarray(contour_grid).save(contour_grid_path)
Image.fromarray(candidate_overlay_grid).save(candidate_overlay_grid_path)
Image.fromarray(candidate_contour_grid).save(candidate_contour_grid_path)

print('Montaje overlays final:', overlay_grid_path)
print('Montaje contornos final:', contour_grid_path)
print('Montaje overlays candidato:', candidate_overlay_grid_path)
print('Montaje contornos candidato:', candidate_contour_grid_path)

fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
axes[0].imshow(overlay_grid)
axes[0].axis('off')
axes[0].set_title('Registro no rigido - resultado final aceptado/conservado')
axes[1].imshow(contour_grid)
axes[1].axis('off')
axes[1].set_title('Contornos resultado final')
plt.show()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
axes[0].imshow(candidate_overlay_grid)
axes[0].axis('off')
axes[0].set_title('Registro no rigido - candidatos antes del filtro')
axes[1].imshow(candidate_contour_grid)
axes[1].axis('off')
axes[1].set_title('Contornos candidatos antes del filtro')
plt.show()


## Lectura del experimento

- Si el candidato sube mucho Dice pero tiene desplazamientos enormes, hay que sospechar.
- Si el resultado final no acepta el candidato, no es fracaso: es una proteccion contra deformaciones poco realistas.
- Si varios sujetos mejoran con desplazamientos moderados, este metodo puede pasar a ser nuestro primer no rigido valido.
- Si solo mejora poco o deforma demasiado, el siguiente candidato natural sera B-spline con regularizacion explicita y una malla de control mas interpretable.
